# Pipeline de Análise Estatística — MOEA puro vs. MOEA + DVL (X%)

Este notebook compara, para cada configuração `(problema, M, avaliações, algoritmo)`, as
**5 variações** do orçamento: **Puro** (MOEA sem DVL) contra **DVL 10%, 25%, 50% e 75%**
(fração das avaliações totais gasta no DVL). O objetivo é verificar se as variações de
DVL trazem ganho estatisticamente significativo sobre o MOEA puro.

Etapas:
1. **Pré-processamento**: leitura de `out/all_results.csv` e mapeamento das variações.
2. **Teste Global (Kruskal-Wallis)**: por configuração, testa se há diferença global
   entre as 5 variações ($p < 0{,}05$).
3. **Pós-hoc (Conover)**: quando o KW rejeita $H_0$, aplica Conover pareado.
4. **Indicação dos melhores (negrito)**: marca a variação com melhor média e todas as
   estatisticamente equivalentes a ela ($p \ge 0{,}05$ no Conover).
5. **Heatmaps**: salva (e opcionalmente exibe) os heatmaps de p-value do Conover por
   configuração, no estilo dos apêndices da dissertação do Artur.
6. **Tabelas LaTeX (Tabelas 19/20)**: médias $\pm$ desvio padrão, com negrito por
   significância, por problema/objetivo.
7. **Vitórias/Empates/Derrotas**: tabela geral e, em especial, a comparação pareada de
   cada variação de DVL **contra o MOEA puro** (ganho/empate/perda).

> Métricas analisadas: **Hipervolume** (maior é melhor) e **Tempo de CPU** (menor é melhor).

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats
import scikit_posthocs as sp
import matplotlib.pyplot as plt

%matplotlib inline

csv_path = "out/all_results.csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"O arquivo {csv_path} não foi encontrado. Execute os experimentos primeiro."
    )

df = pd.read_csv(csv_path)
if "variation" not in df.columns:
    raise ValueError(
        "O CSV está no schema antigo (sem coluna 'variation'). Re-execute o "
        "experimentos.ipynb com o pipeline novo (apague a pasta out/ antes)."
    )

# Considera apenas execuções bem-sucedidas.
if "status" in df.columns:
    df = df[df["status"] == "success"].copy()

print(f"Foram carregados {len(df)} registros de {csv_path}.")

In [ ]:
# Mapeamento das variações para exibição e ordem canônica.
VAR_DISPLAY = {
    "pure_moea": "Puro",
    "dvl_10": "DVL 10%",
    "dvl_25": "DVL 25%",
    "dvl_50": "DVL 50%",
    "dvl_75": "DVL 75%",
}
# Ordem usada em todas as tabelas/heatmaps (Puro sempre na posição 0 = baseline).
variations_list = ["Puro", "DVL 10%", "DVL 25%", "DVL 50%", "DVL 75%"]
PURE_LABEL = "Puro"

ALGO_DISPLAY = {"MOEAD": "MOEA/D", "NSGAII": "NSGA-II", "NSGAIII": "NSGA-III"}
algorithms_order = ["MOEA/D", "NSGA-II", "NSGA-III"]

df["display_variation"] = df["variation"].map(VAR_DISPLAY)
df["display_algorithm"] = df["algorithm"].map(lambda a: ALGO_DISPLAY.get(a, a))

# Número mínimo de execuções por variação para a configuração entrar na análise.
MIN_RUNS = 10

print("Variações:", variations_list)
print("\nContagem por configuração (problema, M, avaliações, algoritmo, variação):")
print(
    df.groupby(["problem", "m", "max_evaluations", "display_algorithm", "display_variation"])
    .size()
    .to_string()
)

In [ ]:
HEATMAP_DIR = "out/heatmaps"

# Cores do heatmap de Conover (igual ao usado pelo Artur):
# 1 (branco) = p >= 0.05 (sem diferença); tons = p < 0.05 / 0.01 / 0.001.
_HEATMAP_CMAP = ["1", "#fb6a4a", "#08306b", "#4292c6", "#c6dbef"]
_HEATMAP_ARGS = {
    "cmap": _HEATMAP_CMAP,
    "linewidths": 0.25,
    "linecolor": "0.5",
    "clip_on": False,
    "square": True,
    "cbar_ax_bbox": [0.85, 0.35, 0.04, 0.3],
}


def run_pipeline_for_metric(
    metric, higher_is_better=True, alpha=0.05, save_heatmaps=True, show_heatmaps=False
):
    """Para cada configuração (problema, M, avaliações, algoritmo), compara as 5
    variações. Retorna (results_rows, ved geral, ved vs. Puro)."""
    configs = df.groupby(["problem", "m", "max_evaluations", "display_algorithm"])

    # V/E/D geral (uma variação "vence" quando é melhor isolada; "empata" quando
    # equivalente ao melhor; "perde" caso contrário).
    wins = {v: 0 for v in variations_list}
    ties = {v: 0 for v in variations_list}
    losses = {v: 0 for v in variations_list}

    # Comparação pareada de cada variação de DVL CONTRA o Puro (ganho/empate/perda).
    dvl_vars = [v for v in variations_list if v != PURE_LABEL]
    pair_better = {v: 0 for v in dvl_vars}
    pair_equal = {v: 0 for v in dvl_vars}
    pair_worse = {v: 0 for v in dvl_vars}

    results_rows = []

    if save_heatmaps:
        os.makedirs(HEATMAP_DIR, exist_ok=True)

    for name, group in configs:
        problem, m, evals, algorithm = name

        data_list, present = [], []
        for v in variations_list:
            vals = group[group["display_variation"] == v][metric].dropna().values
            data_list.append(vals)
            present.append(len(vals))

        # Exige todas as 5 variações com execuções suficientes.
        if any(c < MIN_RUNS for c in present):
            continue

        means = [float(np.mean(d)) for d in data_list]
        stds = [float(np.std(d)) for d in data_list]
        best_idx = int(np.argmax(means) if higher_is_better else np.argmin(means))

        kw_stat, kw_p = stats.kruskal(*data_list)
        if kw_p >= alpha:
            equivalent_idxs = list(range(len(variations_list)))
            pc = pd.DataFrame(1.0, index=variations_list, columns=variations_list)
        else:
            pc = sp.posthoc_conover(data_list)
            pc.index = variations_list
            pc.columns = variations_list
            equivalent_idxs = [
                idx
                for idx in range(len(variations_list))
                if idx == best_idx or pc.iloc[best_idx, idx] >= alpha
            ]

        # V/E/D geral.
        k = len(equivalent_idxs)
        for idx, v in enumerate(variations_list):
            if idx in equivalent_idxs:
                if k == 1:
                    wins[v] += 1
                else:
                    ties[v] += 1
            else:
                losses[v] += 1

        # Pareado vs. Puro.
        pure_idx = variations_list.index(PURE_LABEL)
        for idx, v in enumerate(variations_list):
            if v == PURE_LABEL:
                continue
            p_vs_pure = pc.iloc[pure_idx, idx]
            better_mean = (
                means[idx] > means[pure_idx]
                if higher_is_better
                else means[idx] < means[pure_idx]
            )
            if p_vs_pure < alpha and better_mean:
                pair_better[v] += 1
            elif p_vs_pure < alpha and not better_mean:
                pair_worse[v] += 1
            else:
                pair_equal[v] += 1

        row = {"problem": problem, "m": m, "evals": evals, "algorithm": algorithm}
        for idx, v in enumerate(variations_list):
            row[v] = (means[idx], stds[idx], idx in equivalent_idxs, idx == best_idx)
        results_rows.append(row)

        # Heatmap de p-values do Conover.
        if save_heatmaps or show_heatmaps:
            plt.figure(figsize=(6, 5))
            sp.sign_plot(pc, **_HEATMAP_ARGS)
            plt.title(
                f"Conover - {metric}\n{problem} (M={m}, Aval.={evals}, {algorithm})"
            )
            plt.tight_layout()
            if save_heatmaps:
                fname = f"{metric}_{problem}_m{m}_e{evals}_{algorithm.replace('/', '')}.png"
                plt.savefig(os.path.join(HEATMAP_DIR, fname), dpi=120, bbox_inches="tight")
            if show_heatmaps:
                plt.show()
            else:
                plt.close()

    ved_general = {"wins": wins, "ties": ties, "losses": losses}
    ved_vs_pure = {"better": pair_better, "equal": pair_equal, "worse": pair_worse}
    return results_rows, ved_general, ved_vs_pure

In [ ]:
# Executa a pipeline para Hipervolume (maior é melhor) e Tempo (menor é melhor).
# Os heatmaps de Conover são salvos em out/heatmaps/ (use show_heatmaps=True para
# exibi-los inline — atenção: são muitos).
print("=== HIPERVOLUME ===")
hv_rows, hv_ved, hv_ved_pure = run_pipeline_for_metric(
    "hypervolume", higher_is_better=True, save_heatmaps=True, show_heatmaps=False
)
print(f"{len(hv_rows)} configurações analisadas. Heatmaps em {HEATMAP_DIR}/")

print("\n=== TEMPO DE CPU ===")
time_rows, time_ved, time_ved_pure = run_pipeline_for_metric(
    "cpu_time_seconds", higher_is_better=False, save_heatmaps=True, show_heatmaps=False
)
print(f"{len(time_rows)} configurações analisadas.")

In [ ]:
from IPython.display import display, Markdown


def _index_rows(rows):
    return {(r["problem"], r["m"], r["evals"], r["algorithm"]): r for r in rows}


def _cell(value_tuple, decimals, latex):
    """Formata 'média ± desvio', em negrito se equivalente ao melhor."""
    if value_tuple is None:
        return "--"
    mean, std, is_bold, _is_best = value_tuple
    if latex:
        body = f"{mean:.{decimals}f} \\pm {std:.{decimals}f}"
        return f"$\\mathbf{{{body}}}$" if is_bold else f"${body}$"
    body = f"{mean:.{decimals}f} ± {std:.{decimals}f}"
    return f"**{body}**" if is_bold else body


def build_tables(rows, metric_label, decimals=4):
    """Gera tabelas (LaTeX + Markdown) no estilo das Tabelas 19/20 do Artur:
    uma tabela por (problema, M); linhas = (avaliações × variação); colunas = algoritmos.
    Negrito = melhor média e estatisticamente equivalentes (Conover, p >= 0.05)."""
    idx = _index_rows(rows)
    problems = sorted({r["problem"] for r in rows})
    ms = sorted({r["m"] for r in rows})

    for problem in problems:
        for m in ms:
            evals_list = sorted(
                {r["evals"] for r in rows if r["problem"] == problem and r["m"] == m}
            )
            if not evals_list:
                continue

            # --- Markdown ---
            header = ["Aval.", "Variação"] + algorithms_order
            md = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
            for e in evals_list:
                for v in variations_list:
                    cols = [str(e), v]
                    for algo in algorithms_order:
                        r = idx.get((problem, m, e, algo))
                        cols.append(_cell(r[v] if r else None, decimals, latex=False))
                    md.append("| " + " | ".join(cols) + " |")
            display(Markdown(f"#### {metric_label} — {problem} (M={m})"))
            display(Markdown("\n".join(md)))

            # --- LaTeX ---
            print(f"% --- {metric_label} :: {problem} (M={m}) ---")
            print("\\begin{table}[ht]")
            print("  \\centering")
            print(
                f"  \\caption{{{metric_label} (média $\\pm$ desvio) para {problem}, $M={m}$.}}"
            )
            print(f"  \\label{{tab:{metric_label[:2].lower()}_{problem.lower()}_m{m}}}")
            print("  \\begin{tabular}{ll" + "c" * len(algorithms_order) + "}")
            print("    \\toprule")
            head = ["Aval.", "Variação"] + [f"\\textbf{{{a}}}" for a in algorithms_order]
            print("    " + " & ".join(head) + " \\\\")
            print("    \\midrule")
            for e in evals_list:
                for v in variations_list:
                    cols = [str(e), v.replace("%", "\\%")]
                    for algo in algorithms_order:
                        r = idx.get((problem, m, e, algo))
                        cols.append(_cell(r[v] if r else None, decimals, latex=True))
                    print("    " + " & ".join(cols) + " \\\\")
                if e != evals_list[-1]:
                    print("    \\midrule")
            print("    \\bottomrule")
            print("  \\end{tabular}")
            print("\\end{table}\n")


print("### TABELAS DE HIPERVOLUME (Tabela 19)")
build_tables(hv_rows, "Hipervolume", decimals=4)

print("\n### TABELAS DE TEMPO DE CPU (Tabela 20)")
build_tables(time_rows, "Tempo (s)", decimals=2)

In [ ]:
# ===========================================================================
# 1) Tabela GERAL de Vitórias / Empates / Derrotas (todas as variações entre si)
# ===========================================================================
print("### V/E/D GERAL (cada variação contra todas as outras)")
gen_md = [
    "| Variação | HV: V / E / D | Tempo: V / E / D |",
    "|---|---|---|",
]
for v in variations_list:
    hv_s = f"{hv_ved['wins'][v]} / {hv_ved['ties'][v]} / {hv_ved['losses'][v]}"
    tm_s = f"{time_ved['wins'][v]} / {time_ved['ties'][v]} / {time_ved['losses'][v]}"
    gen_md.append(f"| **{v}** | {hv_s} | {tm_s} |")
display(Markdown("\n".join(gen_md)))

print("\n% --- LaTeX: V/E/D geral ---")
print("\\begin{table}[ht]\n  \\centering")
print("  \\caption{Vitórias/Empates/Derrotas (geral) por variação.}")
print("  \\begin{tabular}{lcc}\n    \\toprule")
print("    Variação & HV (V/E/D) & Tempo (V/E/D) \\\\\n    \\midrule")
for v in variations_list:
    hv_s = f"{hv_ved['wins'][v]} / {hv_ved['ties'][v]} / {hv_ved['losses'][v]}"
    tm_s = f"{time_ved['wins'][v]} / {time_ved['ties'][v]} / {time_ved['losses'][v]}"
    print(f"    {v.replace('%', chr(92)+'%')} & {hv_s} & {tm_s} \\\\")
print("    \\bottomrule\n  \\end{tabular}\n\\end{table}")

# ===========================================================================
# 2) Comparação PAREADA: cada variação de DVL CONTRA o MOEA puro (ganho/empate/perda)
#    -> Este é o requisito principal: "as variações trazem ganho sobre o MOEA puro?"
# ===========================================================================
dvl_vars = [v for v in variations_list if v != PURE_LABEL]
print("\n### DVL vs. MOEA PURO (Ganhos / Empates / Perdas)")
pair_md = [
    "| Variação de DVL | HV: G / E / P | Tempo: G / E / P |",
    "|---|---|---|",
]
for v in dvl_vars:
    hv_s = f"{hv_ved_pure['better'][v]} / {hv_ved_pure['equal'][v]} / {hv_ved_pure['worse'][v]}"
    tm_s = f"{time_ved_pure['better'][v]} / {time_ved_pure['equal'][v]} / {time_ved_pure['worse'][v]}"
    pair_md.append(f"| **{v}** | {hv_s} | {tm_s} |")
display(Markdown("\n".join(pair_md)))

print("\n% --- LaTeX: DVL vs Puro ---")
print("\\begin{table}[ht]\n  \\centering")
print("  \\caption{Ganhos/Empates/Perdas de cada variação de DVL contra o MOEA puro "
      "(teste de Conover, $\\alpha=0{,}05$).}")
print("  \\begin{tabular}{lcc}\n    \\toprule")
print("    Variação de DVL & HV (G/E/P) & Tempo (G/E/P) \\\\\n    \\midrule")
for v in dvl_vars:
    hv_s = f"{hv_ved_pure['better'][v]} / {hv_ved_pure['equal'][v]} / {hv_ved_pure['worse'][v]}"
    tm_s = f"{time_ved_pure['better'][v]} / {time_ved_pure['equal'][v]} / {time_ved_pure['worse'][v]}"
    print(f"    {v.replace('%', chr(92)+'%')} & {hv_s} & {tm_s} \\\\")
print("    \\bottomrule\n  \\end{tabular}\n\\end{table}")

## Como interpretar os resultados para o artigo

A pipeline aplica Kruskal-Wallis (global) seguido de Conover pareado ($\alpha = 0{,}05$)
para comparar, em cada configuração `(problema, M, avaliações, algoritmo)`, o **MOEA puro**
contra as variações **DVL 10%, 25%, 50% e 75%**.

### Hipervolume (maior é melhor)
- A tabela **DVL vs. MOEA puro** (Ganhos/Empates/Perdas) é o resultado central: responde
  diretamente se gastar parte do orçamento no DVL traz ganho estatístico sobre o baseline.
  Um **Ganho** significa que a variação teve média de HV maior *e* diferença significativa
  (Conover $p < 0{,}05$) contra o Puro; **Empate** = sem diferença significativa; **Perda** =
  significativamente pior.
- Espera-se que o benefício do DVL seja maior em orçamentos **restritos** (250–1500), onde a
  semeadura por modelo inverso acelera a convergência; em orçamentos altos (10000) o MOEA
  puro tende a alcançar/superar as variações conforme converge ao Pareto-ótimo.
- A fração ideal de DVL (10/25/50/75%) tende a variar por problema e por número de objetivos;
  as tabelas por problema/$M$ (estilo Tabelas 19/20) mostram, em **negrito**, a melhor média e
  as variações estatisticamente equivalentes a ela.

### Tempo de CPU (menor é melhor)
- Nos benchmarks DTLZ a função objetivo é barata, então o overhead de treinar o MLP a cada
  ponto de referência tende a tornar as variações de DVL **mais lentas** que o MOEA puro.
  Esse custo seria compensado em problemas reais com avaliação cara (ver coluna de
  `model_training_time` vs. `real_evaluation_time` no `experimentos.ipynb`).

### Artefatos gerados
- **Heatmaps** de p-value do Conover por configuração em `out/heatmaps/` (apêndice).
- **Tabelas LaTeX** de HV e Tempo (médias $\pm$ desvio, negrito por significância).
- **Tabelas V/E/D** geral e DVL-vs-Puro, no formato do Capítulo 3 do Artur.